In [1]:
import json
import pandas as pd

from tiu_phi_3_5_mini.data_management import load_train_validation_splits, ActivationsDataSelector, ProbeTrainScenario, \
    train_scenarios_spec_path
from tiu_phi_3_5_mini.evaluation_utils import MetricsForDatasetProbes, evaluate_classifier_performance
from tiu_phi_3_5_mini.logging_setup import create_logger
from tiu_phi_3_5_mini.phi_3_5_constants import dsets_index_path, test_classification_metrics_path, analysis_results_folder
from tiu_phi_3_5_mini.phi_3_5_probe import ProbesForScenario, load_probes_for_dset

In [2]:
logger = create_logger(__name__)

In [3]:
dsets_index_df = pd.read_csv(dsets_index_path, index_col="Idx")
num_dsets = dsets_index_df.shape[0]

In [4]:
train_valid_split_specs = load_train_validation_splits()
data_selector = ActivationsDataSelector(dsets_index_df, train_valid_split_specs)

In [5]:
logger.info(f"loading probe-training scenarios from file {train_scenarios_spec_path}")
with train_scenarios_spec_path.open("r") as f:
    raw_scenarios_specs = json.load(f)
train_scenarios = [ProbeTrainScenario(**raw_scenario_spec) for raw_scenario_spec in raw_scenarios_specs]

2025-02-27 00:14:16,185;__main__;INFO:loading probe-training scenarios from file train_scenarios_spec.json


In [6]:

# first level key identifies the scenario (of one or more datasets) which the probe was trained on;
# second level key identifies the previously-unseen dataset which the probe is tested on
# third level index identifies which train-validation split was used to train a particular group of comparable probes 
#  for a scenario
probes_metrics_on_test_dsets: dict[tuple[int,...], dict[int, list[MetricsForDatasetProbes]]] = {scenario.scenario_key(): {} for scenario in train_scenarios}

In [7]:
for scenario in train_scenarios:
    scenario_key = scenario.scenario_key()
    curr_scenario_probes: list[ProbesForScenario] = []
    for split_variant_idx in range(data_selector.num_split_variants):
        trained_probes = load_probes_for_dset(scenario.result_folder_name, scenario.scenario_name, split_variant_idx)
        curr_scenario_probes.append(trained_probes)
    for test_dset_idx, test_dset_dtls in dsets_index_df.iterrows():
        if test_dset_idx in scenario_key:
            continue  # it wouldn't be a test-set evaluation if the probe was trained/validated on the same dataset
        
        probes_metrics_on_test_dsets[scenario_key][test_dset_idx] = []
        test_dset = data_selector.grab_all_data_for_dset(test_dset_idx)
        for scenario_probes in curr_scenario_probes:
            scenario_metrics_on_curr_test_dset = evaluate_classifier_performance(
                scenario_probes, test_dset.activations, test_dset.truth_labels)
            probes_metrics_on_test_dsets[scenario_key][test_dset_idx].append(scenario_metrics_on_curr_test_dset)

In [8]:
analysis_results_folder.mkdir(exist_ok=True)

In [9]:
serialized_test_metrics = {
    (" ".join(map(str, scenario_id))): {
        str(test_dset_idx): [test_dset_metrics.to_dict() for test_dset_metrics in test_dset_metrics_lst]
        for test_dset_idx, test_dset_metrics_lst in test_dsets_metrics.items()
    } for scenario_id, test_dsets_metrics in probes_metrics_on_test_dsets.items()
}
with test_classification_metrics_path.open("w") as f:
    json.dump(serialized_test_metrics, f, indent=2)